<a href="https://colab.research.google.com/github/saraashraf29/NLP-and-Text-Processing-Basics/blob/main/lab_01_nlp_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 01 — NLP Fundamentals

This lab covers the five topics from the first NLP session:

| # | Topic |
| --- | --- |
| 1 | Regular Expressions |
| 2 | Edit Distances |
| 3 | Text Normalization |
| 4 | Feature Extraction |
| 5 | Text Classification |

Work through each section in order. Read the instructions carefully before writing any code.

> **Tip:** Run the **Setup** cell at the top of each section before attempting the exercise.

---
## Exercise 1 — Regular Expressions

**Skills covered:** `re.findall`, `re.sub`, character classes, quantifiers, word boundaries.

In [1]:
# Setup
import re

### Your task

Given the sentence below, do the following:

1. Find all **hashtags** (e.g. `#Python3`).
2. Find all **numbers** (any sequence of digits).
3. Replace every **phone number** (exactly 11 digits) with the string `'PHONE NUM'`,  
   and replace every other number with the string `'NUM'`.
4. Print the three results.

> **Hint:** Handle the 11-digit phone number *before* you replace shorter numbers,  
> or use a substitution helper function that distinguishes them by length.

In [19]:
sentence = "In 2026, I studied #Python3 and #NLP for 2 months , and I loved it! , this is my phone number call me anytime 01147094321 , "
# 1. Find all hashtags
hashtags = re.findall(r'#\w+', sentence)
print("Hashtags:", hashtags)

# 2. Find all numbers
numbers = re.findall(r'\b\d+\b', sentence)
print("Numbers:", numbers)

# 3. Replace phone numbers then other numbers
cleaned = re.sub(r'\b\d{11}\b', 'PHONE NUM', sentence)
cleaned = re.sub(r'\b\d+\b', 'NUM', cleaned)
print("Cleaned:", cleaned)

Hashtags: ['#Python3', '#NLP']
Numbers: ['2026', '2', '01147094321']
Cleaned: In NUM, I studied #Python3 and #NLP for NUM months , and I loved it! , this is my phone number call me anytime PHONE NUM , 


**Expected output (example):**
```
Hashtags: ['#Python3', '#NLP']
Numbers: ['2026', '2', '01147094321']
Cleaned: In NUM, I studied #Python3 and #NLP for NUM months , and I loved it! , this is my phone number call me anytime PHONE NUM ,
```

---
## Exercise 2 — Edit Distances

**Skills covered:** Levenshtein distance, spelling correction with `min()`.

In [4]:
# Setup — install jellyfish if it is missing
%pip install jellyfish
import jellyfish

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 7.1 MB/s eta 0:00:00


### Your task

The user typed `'houze'`. Use the `jellyfish` library to suggest the closest correct spelling.

1. Calculate the **Levenshtein distance** from `typed_word` to every word in `candidates`.
2. Print each candidate and its distance.
3. Use `min()` to find and print the candidate with the **smallest distance**.

Expected suggestion: `house`

In [20]:
typed_word = "houze"
candidates = ["house", "horse", "mouse"]
for x in candidates:
    dist = jellyfish.levenshtein_distance(typed_word, x)
    print(f"{x}-> {dist}")

#  Print the closest suggestion
closest = min(candidates, key=lambda word: jellyfish.levenshtein_distance(typed_word, word))
print("Suggestion:", closest)

house-> 1
horse-> 2
mouse-> 2
Suggestion: house


**Expected output:**
```
house -> 1
horse -> 2
mouse -> 2
Suggestion: house
```

---
## Exercise 3 — Text Normalization

**Skills covered:** `normalize_text()`, tokenization, stemming with `PorterStemmer`.

In [9]:
# Setup — install nltk if it is missing
# %pip install nltk
import re
from nltk.stem import PorterStemmer

def normalize_text(text):
    """Lowercase, remove punctuation, and collapse whitespace."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # fix extra spaces
    return text

stemmer = PorterStemmer()

### Your task

1. Use `normalize_text()` to clean the sentence below.  
   Expected result: `'students are studying nlp'`
2. Split the cleaned sentence into a list of words.
3. Apply `stemmer.stem()` to every word.
4. Print the cleaned sentence and the list of stems.

In [21]:
sentence = "  Students ARE studying NLP!!!  "
# 1. Normalize
cleaned = normalize_text(sentence)
print("cleaned:",cleaned)

# 2. Tokenize
words = cleaned.split()

# 3 & 4. Stem each word and print
stems = [stemmer.stem(word) for word in words]
print("stems:", stems)

cleaned: students are studying nlp
stems: ['student', 'are', 'studi', 'nlp']


**Expected output:**
```
Cleaned: students are studying nlp
Stems: ['student', 'are', 'studi', 'nlp']
```

---
## Exercise 4 — Feature Extraction (TF-IDF + Cosine Similarity)

**Skills covered:** `TfidfVectorizer`, `cosine_similarity`, document retrieval.

In [14]:
# Setup — install scikit-learn if it is missing
# %pip install scikit-learn
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Your task

Find the document that is most similar to the query.

1. Create a `TfidfVectorizer` with English stop words.
2. Fit it on `documents`.
3. Transform **both** the documents and the query.
4. Calculate the cosine similarity between the query vector and each document vector.
5. Print the document with the **highest similarity score**.

Expected document: `'A kitten is a young cat'`

In [22]:
documents = [
    "A kitten is a young cat",
    "Python is used for data science",
    "The car needs more fuel",
]
query = "young kitten and cat"
# 1 & 2. Create and fit the vectorizer
vectorizer = TfidfVectorizer(stop_words='english')
# 3. Transform documents and query
doc_vectors  = vectorizer.fit_transform(documents)
query_vector = vectorizer.transform([query])
# 4. Compute cosine similarity
scores = cosine_similarity(query_vector, doc_vectors)
# 5. Print the best match
best_index = np.argmax(scores)
print("Most similar document:", documents[best_index])

Most similar document: A kitten is a young cat


---
## Exercise 5 — Text Classification (Sentiment Analysis)

**Skills covered:** `CountVectorizer`, `TfidfVectorizer`, `LogisticRegression`, pipelines, DataFrames.

> **Before you start:** The setup cell below will load the IMDB dataset and train both models.  
> Make sure the dataset file `NLP/datasets/IMDB-Dataset.csv` is available.

In [17]:
# Setup — loads data and trains both models
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

data = pd.read_csv("/content/IMDB-Dataset.csv")
data = data[["review", "sentiment"]].dropna().copy()
data["review"] = data["review"].str.replace("<br />", " ", regex=False)

train_data, test_data = train_test_split(
    data, test_size=0.20, random_state=42, stratify=data["sentiment"]
)
train_data = train_data.reset_index(drop=True)
test_data  = test_data.reset_index(drop=True)

count_model = make_pipeline(
    CountVectorizer(),
    LogisticRegression(max_iter=1000, random_state=42),
)
count_model.fit(train_data["review"], train_data["sentiment"])

tfidf_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=42),
)
tfidf_model.fit(train_data["review"], train_data["sentiment"])

print("Models ready!")

Models ready!


### Your task

Create a small set of movie reviews and compare the two trained models.

1. Write **six short movie reviews** — three positive and three negative.  
   Store them in a list named `student_reviews`.
2. Predict the sentiment of each review with **both** `count_model` and `tfidf_model`.
3. Display the text and both predictions in a **DataFrame** named `student_results`.
4. Answer the reflection questions in the Markdown cell below.

In [23]:
# 1. Write your six movie reviews
student_reviews = [
    # Three positive reviews
    "Absolutely brilliant movie, the acting was spectacular and the plot kept me engaged.",
    "A masterpiece of modern cinema. I loved every second of it.",
    "Highly recommended! The visuals are stunning and the story is heartwarming.",

    # Three negative reviews
    "Terrible movie. The acting was wooden and the plot made no sense.",
    "A complete waste of time. I fell asleep halfway through.",
    "Awful direction and boring dialogue. I regret watching this."
]

# 2. Predict with both models
count_preds = count_model.predict(student_reviews)
tfidf_preds = tfidf_model.predict(student_reviews)

# 3. Build and display the results DataFrame
student_results = pd.DataFrame({
    "review":           student_reviews,
    "count_prediction": count_preds,
    "tfidf_prediction": tfidf_preds,
})

student_results

,review,count_prediction,tfidf_prediction
0,"Absolutely brilliant movie, the acting was spe...",positive,positive
1,A masterpiece of modern cinema. I loved every ...,positive,positive
2,Highly recommended! The visuals are stunning a...,positive,positive
3,Terrible movie. The acting was wooden and the ...,negative,negative
4,A complete waste of time. I fell asleep halfwa...,negative,negative
5,Awful direction and boring dialogue. I regret ...,negative,negative


### Reflection questions

Answer directly in this cell after running your code.

1. **Did the two models disagree on any review?** If yes, which one?

   No, the two models agreed on all six reviews. Because the reviews used very clear and strong sentiment words (like brilliant, masterpiece, terrible, and awful), both models confidently classified them exactly the same way.

2. **Which model gave better predictions for your examples?**

  Both models gave perfect predictions for my specific examples, so they tied. However, in real-world scenarios with longer or more complex reviews, TfidfVectorizer usually performs slightly better because it lowers the weight of common words and highlights the more important, rare words.

3. **Choose one wrong prediction. Why do you think the model made that mistake?**

  Choose one wrong prediction. Why do you think the model made that mistake?
Since neither model made a mistake on these simple examples, let's consider a tricky sentence like: "The movie was not bad at all, I actually liked it."
If a model misclassifies this as negative, it is because both CountVectorizer and TF-IDF rely on a "bag-of-words" approach. The model sees negative words like "not" and "bad" individually and assigns a negative score, failing to understand that "not bad" together means something positive. They struggle with negation, context, and sarcasm.